# Iskanje podatkov PlanetScope in ustvarjanje naročila

Poiščimo satelitske posnetke izbranega območja in ustvarimo naročilo. Naročilo shranimo v zbirko (collection) in ga pošljemo v obdelavo.

## Zahteve

- nameščen Python
- knjižnica requests
- [Planet Account](https://www.planet.com/account/#/)
- API ključ za dostop do Planet API

In [1]:
# Potrebne knjižnice
import os
import json
import math
import requests
from typing import Dict, List, Iterator

## Nastavitev API ključa

Najprej moramo nastaviti naš API ključ, da bomo lahko dostopali do Planet API. To lahko storimo tako, da ga shranimo kot spremenljivko okolja ali pa ga neposredno uporabimo v našem Python skriptu.

In [2]:
# Če vaš Planet API ključ ni nastavljen kot spremenljivka okolja, ga lahko prilepite spodaj
if os.environ.get('PL_API_KEY', ''):
    API_KEY = os.environ.get('PL_API_KEY', '')
else:
    API_KEY = 'PASTE_YOUR_API_KEY_HERE'

In [4]:
# Print API key to verify it was loaded correctly (optional)
print(f'API Key Loaded: {API_KEY[:4]}...{API_KEY[-4:]}')

API Key Loaded: PLAK...1780


## Določitev območja interesa

Območje interesa (**Area of Interest** ali *AOI*) določa geografsko območje, iz katerega želimo pridobiti podatke.

Pri Data API je to lahko preprost pravokotnik (bounding box) s štirimi vogali ali bolj kompleksna oblika, dokler je definicija zapisana v formatu [GeoJSON](http://geojson.org/).

V tem primeru bomo uporabili preprost pravokotnik. Za enostavno ustvarjanje lahko uporabimo [geojson.io](http://geojson.io/), kjer hitro narišemo obliko in dobimo GeoJSON zapis:

![geojsonio.png](./slike/geojsonio.png)

Za zahtevo Data API potrebujemo samo objekt **geometry**, ki ga lahko preberemo iz GeoJSON datoteke. Ta objekt vsebuje koordinate, ki definirajo naše AOI. Na primer, če imamo GeoJSON datoteko z imenom `aoi.geojson`, lahko preberemo geometrijo.

In [5]:
# Ime datoteke z AOI
aoi_file = './podatki/kranj_aoi.geojson'
with open(aoi_file) as f:
    geojson_data = json.load(f)
geojson_geometry = geojson_data['features'][0]['geometry']

In [6]:
# Izpis geometrije AOI, da preverite, ali je pravilno prebrana
print(json.dumps(geojson_geometry, indent=2, ensure_ascii=False))

{
  "coordinates": [
    [
      [
        14.338274979185115,
        46.234310509304606
      ],
      [
        14.384337227889091,
        46.234310509304606
      ],
      [
        14.384337227889091,
        46.262948793445105
      ],
      [
        14.338274979185115,
        46.262948793445105
      ],
      [
        14.338274979185115,
        46.234310509304606
      ]
    ]
  ],
  "type": "Polygon"
}


## Nastavitve

In [7]:
# Nastavitve iskanja in naročila
ORDER_NAME = "Kranj 2026 test"  # Poimenujte naročilo po želji
ITEM_TYPE = "PSScene"
PRODUCT_BUNDLE = "analytic_8b_udm2"   # 8-band + UDM2 quality mask (alternative: analytic_8b_sr_udm2)
START_DATE = "2026-03-01T00:00:00Z"
END_DATE = "2026-03-05T23:59:59Z"
MAX_CLOUD_COVER = 0.20             # 20%
BATCH_SIZE = 500                   # Orders API supports up to 500 items/order
CLIP_TO_AOI = False  # default: no clipping

In [8]:
# API endpointa za iskanje posnetkov in ustvarjanje naročil
DATA_SEARCH_URL = "https://api.planet.com/data/v1/quick-search"
ORDERS_URL = "https://api.planet.com/compute/ops/orders/v2"

### API povezave in avtentikacija

V tej sekciji določimo URL-je storitev in inicializiramo HTTP sejo z API ključem.

In [9]:
# Inicializacija HTTP seje in avtentikacije za vse API klice
SESSION = requests.Session()
SESSION.headers.update({"Content-Type": "application/json"})
SESSION.auth = (API_KEY, "")

## Pomožne funkcije

Funkcije spodaj pripravijo filter iskanja, izvajajo straničeno branje rezultatov in ustvarijo naročila.

In [10]:
# Sestava kombiniranega filtra za AOI, časovno obdobje in oblačnost
def build_search_filter(aoi: Dict, start_date: str, end_date: str, max_cloud: float) -> Dict:
    # Sestavimo kombinirani filter: geometrija + datum + oblačnost.
    return {
        "type": "AndFilter",
        "config": [
            {
                "type": "GeometryFilter",
                "field_name": "geometry",
                "config": aoi
            },
            {
                "type": "DateRangeFilter",
                "field_name": "acquired",
                "config": {
                    "gte": start_date,
                    "lte": end_date
                }
            },
            {
                "type": "RangeFilter",
                "field_name": "cloud_cover",
                "config": {
                    "lte": max_cloud
                }
            }
        ]
    }

In [11]:
# Iskanje posnetkov preko Quick Search API z avtomatskim prehajanjem po straneh
def search_items(item_type: str, search_filter: Dict, page_size: int = 250) -> Iterator[Dict]:
    body = {
        "item_types": [item_type],
        "filter": search_filter
    }

    # Začetni klic z naraščajočim časovnim vrstnim redom.
    url = f"{DATA_SEARCH_URL}?_sort=acquired%20asc&_page_size={page_size}"
    resp = SESSION.post(url, json=body)
    resp.raise_for_status()
    data = resp.json()

    # Iteriramo čez vse strani rezultatov, dokler API vrača _next povezavo.
    while True:
        for feature in data.get("features", []):
            yield feature

        next_url = data.get("_links", {}).get("_next")
        if not next_url:
            break

        next_resp = SESSION.get(next_url)
        next_resp.raise_for_status()
        data = next_resp.json()

In [12]:
# Razdeli seznam ID-jev na manjše pakete velikosti size
def chunked(seq: List[str], size: int) -> Iterator[List[str]]:
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

In [13]:
def create_hosted_order(order_name: str, item_ids: List[str], item_type: str, product_bundle: str, aoi: Dict, clip: bool) -> Dict:
    payload = {
        "name": order_name,
        "source_type": "scenes",
        "products": [
            {
                "item_ids": item_ids,
                "item_type": item_type,
                "product_bundle": product_bundle
            }
        ],
        "hosting": {
            "sentinel_hub": {
                "create_configuration": True
            }
        }
    }

    # add clipping only if enabled
    if clip:
        payload["tools"] = [
            {
                "clip": {
                    "aoi": aoi
                }
            }
        ]

    resp = SESSION.post(ORDERS_URL, json=payload)
    if not resp.ok:
        print(f"Planet API error {resp.status_code}")
        try:
            print(json.dumps(resp.json(), indent=2, ensure_ascii=False))
        except ValueError:
            print(resp.text)
        resp.raise_for_status()
    return resp.json()

## Iskanje posnetkov

In [14]:
# Filter za iskanje
search_filter = build_search_filter(geojson_geometry, START_DATE, END_DATE, MAX_CLOUD_COVER)

In [15]:
# Iskanje posnetkov v katalogu in priprava seznama ID-jev
print("Searching Planet catalog...")
# Iz iskalnika preberemo vse najdene feature-je in izluščimo njihove ID-je.
features = list(search_items(ITEM_TYPE, search_filter))
item_ids = [f["id"] for f in features]

Searching Planet catalog...


In [16]:
# Print item IDs to verify they were collected correctly
print(f"Found {len(item_ids)} items: {item_ids[:5]}...")

Found 9 items: ['20260301_095735_02_255f', '20260304_102640_67_252e', '20260304_102642_93_252e', '20260304_102747_92_253b', '20260304_102750_20_253b']...


In [17]:
# Filtriranje na posnetke, za katere ima račun pravico do izbranega product bundle
required_permissions_by_bundle = {
    "analytic_udm2": {"assets.ortho_analytic_4b:download", "assets.ortho_analytic_4b_xml:download", "assets.ortho_udm2:download"},
    "analytic_8b_udm2": {"assets.ortho_analytic_8b:download", "assets.ortho_analytic_8b_xml:download", "assets.ortho_udm2:download"},
    "analytic_8b_sr_udm2": {"assets.ortho_analytic_8b_sr:download", "assets.ortho_analytic_8b_sr_xml:download", "assets.ortho_udm2:download"},
}

required_permissions = required_permissions_by_bundle.get(PRODUCT_BUNDLE, set())

if required_permissions:
    accessible_features = []
    inaccessible = []

    for f in features:
        perms = set(f.get("_permissions") or f.get("permissions") or [])
        if required_permissions.issubset(perms):
            accessible_features.append(f)
        else:
            missing = sorted(required_permissions - perms)
            inaccessible.append((f.get("id"), missing))

    features = accessible_features
    item_ids = [f["id"] for f in features]

    print(f"Accessible for {PRODUCT_BUNDLE}: {len(item_ids)} of {len(item_ids) + len(inaccessible)}")

    if inaccessible:
        print("First inaccessible examples:")
        for scene_id, missing in inaccessible[:5]:
            print(f"- {scene_id} | missing permissions: {missing}")
else:
    print(f"No permission rule configured for PRODUCT_BUNDLE={PRODUCT_BUNDLE}; skipping access filter.")

Accessible for analytic_8b_udm2: 0 of 9
First inaccessible examples:
- 20260301_095735_02_255f | missing permissions: ['assets.ortho_analytic_8b:download', 'assets.ortho_analytic_8b_xml:download', 'assets.ortho_udm2:download']
- 20260304_102640_67_252e | missing permissions: ['assets.ortho_analytic_8b:download', 'assets.ortho_analytic_8b_xml:download', 'assets.ortho_udm2:download']
- 20260304_102642_93_252e | missing permissions: ['assets.ortho_analytic_8b:download', 'assets.ortho_analytic_8b_xml:download', 'assets.ortho_udm2:download']
- 20260304_102747_92_253b | missing permissions: ['assets.ortho_analytic_8b:download', 'assets.ortho_analytic_8b_xml:download', 'assets.ortho_udm2:download']
- 20260304_102750_20_253b | missing permissions: ['assets.ortho_analytic_8b:download', 'assets.ortho_analytic_8b_xml:download', 'assets.ortho_udm2:download']


In [18]:
# optional: print a few matches
for f in features[:5]:
    props = f.get("properties", {})
    print(f"- {f['id']} | acquired={props.get('acquired')} | cloud_cover={props.get('cloud_cover')}")

## Ustvarjanje naročil

Najdene posnetke razdelimo v pakete in za vsak paket ustvarimo ločeno naročilo.

In [19]:
# Naročilo lahko terja veliko enot procesiranja, bolje je obrezati na AOI
CLIP_TO_AOI = True  # omogoči izrezovanje v območju AOI za hitrejšo obdelavo in manjše prenose

In [20]:
# po potrebi ustvarite eno ali več naročil
orders = []
num_batches = math.ceil(len(item_ids) / BATCH_SIZE)
print(f"Creating orders in batches of {BATCH_SIZE} (total {num_batches} batches)...")

Creating orders in batches of 500 (total 0 batches)...


In [21]:
# Ustvarjanje enega ali več naročil glede na velikost batcha
for idx, batch in enumerate(chunked(item_ids, BATCH_SIZE), start=1):
    # Če je rezultatov več kot BATCH_SIZE, jih razdelimo na več naročil.
    batch_name = f"{ORDER_NAME}-part-{idx:02d}-of-{num_batches:02d}" if num_batches > 1 else ORDER_NAME
    print(f"Creating order: {batch_name} ({len(batch)} items)")
    order = create_hosted_order(
        batch_name,
        batch,
        ITEM_TYPE,
        PRODUCT_BUNDLE,
        geojson_geometry,
        CLIP_TO_AOI,
    )
    orders.append(order)

    order_id = order.get("id")
    state = order.get("state")
    collection_id = (
        order.get("hosting", {})
                .get("sentinel_hub", {})
                .get("collection_id")
    )

    print(f"  order_id: {order_id}")
    print(f"  state: {state}")
    print(f"  collection_id: {collection_id}")

## Povzetek rezultatov

Na koncu izpišemo vsa ustvarjena naročila in njihovo trenutno stanje.

In [22]:
# Končni izpis ustvarjenih naročil in njihovih stanj
print("\nDone.")
print("Created orders:")
for order in orders:
    print(f"- {order.get('id')} | {order.get('name')} | {order.get('state')}")


Done.
Created orders:
